In [1]:
from datetime import date, datetime
from dateutil.relativedelta import relativedelta
from functools import partial
from tenacity import retry, stop_after_delay, wait_fixed, retry_if_exception_type
from eutils import EutilsNCBIError, EutilsRequestError

import os as os
import pandas as pd 
import numpy as np 

from metapub import PubMedFetcher 
from metapub import pubmedcentral
import logging
import time

from concurrent.futures import ThreadPoolExecutor, as_completed
import csv

In [3]:
os.chdir("./Query_to_full_text")


file_handler = logging.FileHandler('PMC.log', mode='w')
formatter = logging.Formatter('%(asctime)s - %(levelname)s - %(message)s')
file_handler.setFormatter(formatter)
logging.getLogger().addHandler(file_handler)

# Initialize the fetcher for PubMed and Crossref
fetcher = PubMedFetcher()


# Retry logic for handling communication errors
retry_on_communication_error = partial(
    retry,
    stop=stop_after_delay(10),  # Maximum 10 seconds wait.
    wait=wait_fixed(0.4),  # Wait 400ms between retries
    retry=retry_if_exception_type((Exception,))  # Use a tuple for exceptions
)


In [4]:

def read_query_from_file(filename):
    """Read and clean query from a file."""
    try:
        with open(filename, 'r') as file:
            query = file.read()
        return query
    except FileNotFoundError:
        logging.error(f"The file '{filename}' was not found.")
        return ""
    except Exception as e:
        logging.error(f"An error occurred while reading the query file: {e}")
        return ""

@retry_on_communication_error()
def get_list(query):
    """Retrieve all PMIDs for a given query using the PubMedFetcher."""
    num_of_articles = 500
    start_index = 0
    pmids = []
    while True:
        pmid_batch = fetcher.pmids_for_query(query,
                                            retstart=start_index,
                                            retmax=num_of_articles,
                                            pmc_only=True)
        pmids.extend(pmid_batch)
        start_index = len(pmids)
        if len(pmid_batch) < num_of_articles:
            break
    return pmids

@retry_on_communication_error()
def fetch_pmids_over_period(query_file, start="2000-01-01", stop=None):
    """Fetch PMIDs over a specified period using a query read from a file."""
    query = read_query_from_file(query_file)
    if not query:
        logging.error("Failed to read query.")
        return np.array([])

    if stop is None:
        stop = datetime.now().strftime("%Y-%m-%d")

    start_date_str = start
    pmid_list = []

    while True:
        if date.fromisoformat(start_date_str) <= date.fromisoformat("2002-07-01"):
            month_interval = 6
        elif date.fromisoformat(start_date_str) <= date.fromisoformat("2005-11-01"):
            month_interval = 5
        elif date.fromisoformat(start_date_str) <= date.fromisoformat("2009-11-01"):
            month_interval = 4
        elif date.fromisoformat(start_date_str) <= date.fromisoformat("2011-10-01"):
            month_interval = 3
        elif date.fromisoformat(start_date_str) <= date.fromisoformat("2023-01-01"):
            month_interval = 2
        else:
            month_interval = 4

        next_start = date.fromisoformat(start_date_str) + relativedelta(months=month_interval)
        end_date = (next_start - relativedelta(days=1))
        end_date_str = end_date.strftime('%Y-%m-%d')

        date_str = f'''(("{start_date_str}"[Date - Publication] : "{end_date_str}"[Date - Publication]) '''
        pmids = get_list(date_str + query)
        pmid_list.extend(pmids)
        start_date_str = next_start.strftime('%Y-%m-%d')
        if next_start >= date.fromisoformat(stop):
            break

    # Remove duplicates by converting to a set, then back to a list
    pmid_clean_list = list(set(pmid_list))
    logging.info(f"Total PMIDs fetched: {len(pmid_clean_list)}")

    return np.array(pmid_clean_list)

def save_pmids(pmid_array, directory="PMID_lists"):
    """Save PMIDs to both a text file and a NumPy binary file."""
    # Ensure the directory exists
    os.makedirs(directory, exist_ok=True)

    # Create a date tag for the filename
    date_tag = datetime.now().isoformat()[:10]

    # File paths
    txt_file_path = os.path.join(directory, f'pmids_{date_tag}.txt')
    npy_file_path = os.path.join(directory, f'pmids_{date_tag}.npy')

    # Save PMIDs to a text file
    np.savetxt(txt_file_path, pmid_array, fmt='%s', delimiter=",")
    logging.info(f"PMIDs saved to text file: {txt_file_path}")

    # Save PMIDs to a NumPy binary file
    np.save(npy_file_path, pmid_array)
    logging.info(f"PMIDs saved to binary file: {npy_file_path}")


@retry_on_communication_error
def fetch_pmcid(pmid):
    try:
        pmc = pubmedcentral.get_pmcid_for_otherid(pmid)
        return pmc
    except (CommunicationError, ConnectionError) as e:
        logging.error(f"Error: API request failed for {pmid}: {e}")
        return None
    except Exception as e:
        logging.error(f"Unexpected error for {pmid}: {e}")
        return None

def get_pmcid_for_otherid(pmid_clean_list):
    PMCIDs = []
    with ThreadPoolExecutor(max_workers=10) as executor:  # Adjust max_workers based on needs
        future_to_pmid = {executor.submit(fetch_pmcid, pmid): pmid for pmid in pmid_clean_list}
        for future in as_completed(future_to_pmid):
            pmid = future_to_pmid[future]
            try:
                pmc = future.result()
                PMCIDs.append(pmc)
            except Exception as e:
                logging.error(f"Error processing PMID {pmid}: {e}")
                PMCIDs.append(None)
    return PMCIDs

def filter_oa_database(oa_file_list, pmc_ids_filename):
    """
    Filters based on the csv database list of PMCs that are available for full_text mining and writes them  to CSV and txt.

    Parameters:
    oa_file_list (str): Filename of the CSV containing the OA file list.
    pmc_ids_filename (str): Filename of the CSV containing the PMC IDs.
    """


    # Read CSV files
    oa_file_list_df = pd.read_csv(oa_file_list)
    pmc_ids_df = pd.read_csv(pmc_ids_filename)

    # Extract PMC ID list from the DataFrame
    pmc_id_list = pmc_ids_df.iloc[:, 0].tolist()

    # Filter oa_database based on PMC ID list
    filtered_oa_database = oa_file_list_df[oa_file_list_df["Accession ID"].isin(pmc_id_list)]

    # Open a text file to write
    with open('full_text_pmc.txt', 'w') as file:
        for item in filtered_oa_database["Accession ID"]:
            file.write(str(item) + '\n')


    # Save the filtered DataFrame to a new CSV file
    filtered_oa_database.to_csv("full_text_data.csv", index=False)

    return filtered_oa_database


In [5]:
def main():
    # Step 1: Read the query from a file
    query_file = "query" 
    query = read_query_from_file(query_file)
    if not query:
        logging.error("Query reading failed. Exiting.")
        return

    # Step 2: Fetch PMIDs over a specified period
    start_date = "2000-01-01"
    stop_date = None  # Will default to the current date if None
    pmid_array = fetch_pmids_over_period(query_file, start=start_date, stop=stop_date)
    if pmid_array.size == 0:
        logging.error("No PMIDs fetched. Exiting.")
        return

    # Step 3: Save the PMIDs to files
    save_pmids(pmid_array)

    # Step 4: Retrieve PMCIDs for the fetched PMIDs
    pmc_id_list = get_pmcid_for_otherid(pmid_array)
    pmc_ids_filename = "PMCIDS.csv"
    pd.DataFrame(pmc_id_list, columns=["PMCID"]).to_csv(pmc_ids_filename, index=False)
    logging.info(f"PMCIDs saved to {pmc_ids_filename}")

    # Step 5: Filter the OA database using the retrieved PMCIDs
    oa_file_list = "oa_file_list.csv"  # OA file list CSV filename
    pmc_ids_filename = "PMCIDS.csv"
    filtered_df = filter_oa_database(oa_file_list, pmc_ids_filename)

    logging.info("OA database filtering completed.")
    return filtered_df

# Call the main function to execute the workflow
if __name__ == "__main__":
    main()


2024-08-21 12:35:55 LAPTOP-S8N3C7A8 root[18032] INFO Total PMIDs fetched: 298
2024-08-21 12:35:56 LAPTOP-S8N3C7A8 root[18032] INFO PMIDs saved to text file: PMID_lists\pmids_2024-08-21.txt
2024-08-21 12:35:56 LAPTOP-S8N3C7A8 root[18032] INFO PMIDs saved to binary file: PMID_lists\pmids_2024-08-21.npy
2024-08-21 12:36:09 LAPTOP-S8N3C7A8 root[18032] INFO PMCIDs saved to PMCIDS.csv
2024-08-21 12:36:44 LAPTOP-S8N3C7A8 root[18032] INFO OA database filtering completed.


In [11]:
!curl -s "https://www.ncbi.nlm.nih.gov/research/bionlp/RESTful/pmcoa.cgi/BioC_json/PMC10759277/unicode" > "PMC10759277.json"


In [25]:
%%bash

mkdir -p ./Full_text_jsons

while IFS= read -r PMCID; do
    url="https://www.ncbi.nlm.nih.gov/research/bionlp/RESTful/pmcoa.cgi/BioC_json/${PMCID}/unicode"
    curl -s "${url}" > "./Full_text_jsons/${PMCID}.json"
done < full_text_pmc.txt



<3>WSL (10) ERROR: CreateProcessEntryCommon:505: execvpe /bin/bash failed 2
<3>WSL (10) ERROR: CreateProcessEntryCommon:508: Create process not expected to return


CalledProcessError: Command 'b'\nwhile IFS= read -r PMCID; do\n    url="https://www.ncbi.nlm.nih.gov/research/bionlp/RESTful/pmcoa.cgi/BioC_json/${PMCID}/unicode"\n    curl -s "${url}" > "${PMCID}.json"\ndone < full_text_pmc.txt\n\n'' returned non-zero exit status 1.

In [77]:
!powershell -Command "mkdir -p ./Full_text_jsons; Get-Content full_text_pmc.txt | ForEach-Object { Invoke-RestMethod -Uri ('https://www.ncbi.nlm.nih.gov/research/bionlp/RESTful/pmcoa.cgi/BioC_json/' + $_ + '/unicode') -OutFile ('./Full_text_jsons/' + $_ + '.json') }"

^C




    Directory: C:\Users\mkha1\Desktop\GQP\pmc fulltxt ipynb


Mode                 LastWriteTime         Length Name                                                                 
----                 -------------         ------ ----                                                                 
d-----        20/08/2024     23:45                Full_text_jsons                                                      




Invoke-RestMethod : Could not find a part of the path 'C:\Users\mkha1\Desktop\GQP\pmc fulltxt 
ipynb\Full_text_jsons\PMC10857618.json'.
At line:1 char:78
+ ... ch-Object { Invoke-RestMethod -Uri ('https://www.ncbi.nlm.nih.gov/res ...
+                 ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
    + CategoryInfo          : NotSpecified: (:) [Invoke-RestMethod], DirectoryNotFoundException
    + FullyQualifiedErrorId : System.IO.DirectoryNotFoundException,Microsoft.PowerShell.Commands.InvokeRestMethodComma 
   nd
 
Invoke-RestMethod : Could not find a part of the path 'C:\Users\mkha1\Desktop\GQP\pmc fulltxt 
ipynb\Full_text_jsons\PMC10858073.json'.
At line:1 char:78
+ ... ch-Object { Invoke-RestMethod -Uri ('https://www.ncbi.nlm.nih.gov/res ...
+                 ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
    + CategoryInfo          : NotSpecified: (:) [Invoke-RestMethod], DirectoryNotFoundException
    + FullyQualifiedErrorId : System.IO.DirectoryNotFoundExc